In [ ]:
!pip install pyspark==3.5.0
!pip install spark-nlp==5.5.3

# Introduction

## Corpus et pipeline d'extraction

Le corpus est composé de 14 romans du XIXe siècle répartis en deux mouvements littéraires : le **Romantisme** et le **Réalisme**.
Les textes ont été téléchargés depuis [Project Gutenberg](https://www.gutenberg.org/).

In [42]:
import sparknlp
import requests
import re
import time
import os
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml import Pipeline
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from pyspark.sql.functions import col, size, udf, explode, lower, count, posexplode, floor
from pyspark.sql.functions import round as spark_round, sum as spark_sum, row_number
from pyspark.sql.window import Window
import plotly.express as px

spark = sparknlp.start()
print("Spark-NLP:", sparknlp.version())
print("Spark:", spark.version)

Spark-NLP: 5.5.3
Spark: 3.5.0


In [43]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/projet_BD/DATA")
RAW_DIR = DATA_DIR / "raw_texts"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Drive monté !")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive monté !


## Téléchargement des textes

...

In [57]:
DATA_DIR = Path("/content/drive/MyDrive/projet_BD/DATA")
RAW_DIR = DATA_DIR / "raw_texts"
RAW_DIR.mkdir(parents=True, exist_ok=True)

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/{id}/pg{id}.txt"

CORPUS = [
    {"author": "Balzac",     "title": "Pere_Goriot",          "id": 1237,  "movement": "romantic", "lang": "en"},
    {"author": "Balzac",     "title": "Eugenie_Grandet",       "id": 1608,  "movement": "romantic", "lang": "en"},
    {"author": "Dumas",      "title": "Monte_Cristo",          "id": 17989, "movement": "romantic", "lang": "en"},
    {"author": "Stendhal",   "title": "Rouge_et_Noir",         "id": 798,   "movement": "romantic", "lang": "en"},
    {"author": "Sand",       "title": "Indiana",               "id": 5837,  "movement": "romantic", "lang": "en"},
    {"author": "Bronte_C",   "title": "Jane_Eyre",             "id": 1260,  "movement": "romantic", "lang": "en"},
    {"author": "Bronte_E",   "title": "Wuthering_Heights",     "id": 768,   "movement": "romantic", "lang": "en"},
    {"author": "Scott",      "title": "Ivanhoe",               "id": 82,    "movement": "romantic", "lang": "en"},
    {"author": "Flaubert",   "title": "Madame_Bovary",         "id": 2413,  "movement": "realist",  "lang": "en"},
    {"author": "Zola",       "title": "Germinal",              "id": 5711,  "movement": "realist",  "lang": "en"},
    {"author": "Zola",       "title": "Assommoir",             "id": 8600,  "movement": "realist",  "lang": "en"},
    {"author": "Dickens",    "title": "Great_Expectations",    "id": 1400,  "movement": "realist",  "lang": "en"},
    {"author": "Dickens",    "title": "Oliver_Twist",          "id": 730,   "movement": "realist",  "lang": "en"},
    {"author": "Maupassant", "title": "Bel_Ami",               "id": 4598,  "movement": "realist",  "lang": "en"},
]

def clean_gutenberg(text):
    start = re.search(r"\*\*\* START OF TH(IS|E) PROJECT GUTENBERG", text)
    end   = re.search(r"\*\*\* END OF TH(IS|E) PROJECT GUTENBERG",   text)
    if start and end:
        text = text[start.end():end.start()]
    return text.strip()

def download_all():
    for book in CORPUS:
        path = RAW_DIR / book["author"] / f"{book['title']}.txt"
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists():
            print(f"[cache] {book['title']}")
            continue
        url = GUTENBERG_URL.format(id=book["id"])
        response = requests.get(url, timeout=30)
        text = clean_gutenberg(response.text)
        path.write_text(text, encoding="utf-8")
        print(f"[ok] {book['title']} — {len(text):,} caractères")
        time.sleep(1)

download_all()

[cache] Pere_Goriot
[cache] Eugenie_Grandet
[cache] Monte_Cristo
[cache] Rouge_et_Noir
[cache] Indiana
[cache] Jane_Eyre
[cache] Wuthering_Heights
[cache] Ivanhoe
[cache] Madame_Bovary
[cache] Germinal
[cache] Assommoir
[cache] Great_Expectations
[cache] Oliver_Twist
[cache] Bel_Ami


In [58]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

rows = []
for author_dir in RAW_DIR.iterdir():
    if not author_dir.is_dir():
        continue
    author = author_dir.name
    for txt_file in author_dir.glob("*.txt"):
        title = txt_file.stem
        text = txt_file.read_text(encoding="utf-8")
        movement = next(b["movement"] for b in CORPUS if b["title"] == title)
        lang = next(b["lang"] for b in CORPUS if b["title"] == title)
        rows.append((author, title, movement, lang, text, len(text), len(text.split())))

schema = StructType([
    StructField("author",   StringType(), True),
    StructField("title",    StringType(), True),
    StructField("movement", StringType(), True),
    StructField("lang",     StringType(), True),
    StructField("text",     StringType(), True),
    StructField("n_chars",  LongType(),   True),
    StructField("n_words",  LongType(),   True),
])

df = spark.createDataFrame(rows, schema=schema)
df.select("author", "title", "movement", "lang", "n_chars", "n_words").show()

+----------+------------------+--------+----+-------+-------+
|    author|             title|movement|lang|n_chars|n_words|
+----------+------------------+--------+----+-------+-------+
|    Balzac|       Pere_Goriot|romantic|  en| 571261| 103598|
|    Balzac|   Eugenie_Grandet|romantic|  en| 348752|  66578|
|     Dumas|      Monte_Cristo|romantic|  en| 727540| 121587|
|  Stendhal|     Rouge_et_Noir|romantic|  en|1008461| 171925|
|      Sand|           Indiana|romantic|  en|  79911|  14032|
|  Bronte_C|         Jane_Eyre|romantic|  en|1022418| 185396|
|  Bronte_E| Wuthering_Heights|romantic|  en| 645637| 115949|
|     Scott|           Ivanhoe|romantic|  en|1103759| 191646|
|  Flaubert|     Madame_Bovary| realist|  en| 652156| 115667|
|      Zola|          Germinal| realist|  en|1011885| 164610|
|      Zola|         Assommoir| realist|  en| 870946| 155848|
|   Dickens|Great_Expectations| realist|  en| 994646| 184482|
|   Dickens|      Oliver_Twist| realist|  en| 893187| 157973|
|Maupass

In [46]:
from pyspark.sql.functions import col, size, udf
from pyspark.sql.types import LongType
from pyspark.sql.functions import round as spark_round

def compute_syllables(text):
    if text is None:
        return 0
    vowels = "aeiouyAEIOUY"
    count = 0
    prev_vowel = False
    for char in text:
        is_vowel = char in vowels
        if is_vowel and not prev_vowel:
            count += 1
        prev_vowel = is_vowel
    return count

syllable_udf = udf(compute_syllables, LongType())

df_fk = df_annotated \
    .withColumn("n_sentences", size(col("sentences"))) \
    .withColumn("n_syllables", syllable_udf(col("text"))) \
    .withColumn("flesch_kincaid", spark_round(
        206.835
        - 1.015 * (col("n_words") / col("n_sentences"))
        - 84.6  * (col("n_syllables") / col("n_words")), 2)) \
    .withColumn("kandel_moles", spark_round(
        207
        - 1.015 * (col("n_words") / col("n_sentences"))
        - 73.6  * (col("n_syllables") / col("n_words")), 2))

df_fk.select("author", "title", "movement", "flesch_kincaid", "kandel_moles").show()

+----------+------------------+--------+--------------+------------+
|    author|             title|movement|flesch_kincaid|kandel_moles|
+----------+------------------+--------+--------------+------------+
|    Balzac|       Pere_Goriot|romantic|         66.82|       83.39|
|    Balzac|   Eugenie_Grandet|romantic|         69.31|       85.27|
|     Dumas|      Monte_Cristo|romantic|         60.69|       77.95|
|  Stendhal|     Rouge_et_Noir|romantic|         61.09|       78.27|
|      Sand|           Indiana|romantic|         69.37|       86.04|
|  Bronte_C|         Jane_Eyre|romantic|         64.48|       81.16|
|  Bronte_E| Wuthering_Heights|romantic|         64.96|       81.62|
|     Scott|           Ivanhoe|romantic|         50.23|       67.54|
|  Flaubert|     Madame_Bovary| realist|         63.62|        80.4|
|      Zola|          Germinal| realist|         57.63|       75.32|
|      Zola|         Assommoir| realist|         65.66|       82.27|
|   Dickens|Great_Expectations| re

## Sauvegarde en Parquet

Nous sauvegardons le DataFrame au format **Parquet**, un format orienté  colonne qui offre de meilleures performances en lecture que CSV ou JSON, notamment grâce à la compression et au pushdown de prédicats.

In [47]:
df.write.mode("overwrite").parquet(str(DATA_DIR / "corpus_raw.parquet"))
print("sauvegardé !")

sauvegardé !


# Annotation avec Spark-NLP

## Pipeline d'annotation

Nous utilisons Spark-NLP 5.5.3 avec PySpark 3.5.0. Le pipeline applique trois étapes successives sur chaque roman en parallèle :

1. **DocumentAssembler** — convertit le texte brut en document Spark-NLP
2. **SentenceDetector** — découpe le document en phrases
3. **Tokenizer** — découpe chaque phrase en mots

In [48]:
document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

sentence_detector = SentenceDetector() \
    .setInputCols(["document"]) \
    .setOutputCol("sentences")

tokenizer = Tokenizer() \
    .setInputCols(["sentences"]) \
    .setOutputCol("tokens")

pipeline = Pipeline(stages=[
    document_assembler,
    sentence_detector,
    tokenizer
])

model = pipeline.fit(df)
df_annotated = model.transform(df)

df_annotated.write.mode("overwrite").parquet(str(DATA_DIR / "corpus_annotated.parquet"))
print("annoté et sauvegardé !")

annoté et sauvegardé !


# Analyse Stylométrique

## Indices de lisibilité

Les indices de lisibilité mesurent quantitativement la difficulté de lecture  d'un texte. Nous calculons deux indices : **Flesch-Kincaid** (FK) et
**Kandel-Moles** (KM), tous deux basés sur la longueur des phrases et lacomplexité des mots.

### Flesch-Kincaid

La formule de Flesch-Kincaid est :

$$FK = 206.835 - 1.015 \times \frac{mots}{phrases} - 84.6 \times \frac{syllabes}{mots}$$

Un score élevé indique un texte facile à lire. Les scores varient généralement entre 0 (très difficile) et 100 (très facile).

In [49]:
import plotly.express as px

df_plot = df_fk.select("author", "title", "movement", "flesch_kincaid", "kandel_moles").toPandas()

# Moyenne par auteur
df_avg = df_plot.groupby(["author", "movement"], as_index=False).agg(
    flesch_kincaid=("flesch_kincaid", "mean"),
    kandel_moles=("kandel_moles", "mean")
).round(2)

fig = px.bar(
    df_avg.sort_values("flesch_kincaid"),
    x="author",
    y="flesch_kincaid",
    color="movement",
    title="Indice de lisibilité Flesch-Kincaid par auteur (moyenne)",
    labels={
        "flesch_kincaid": "Score Flesch-Kincaid",
        "author": "Auteur",
        "movement": "Mouvement littéraire"
    },
    color_discrete_map={"romantic": "#e07b54", "realist": "#4a90d9"}
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [50]:
fig2 = px.bar(
    df_avg.sort_values("kandel_moles"),
    x="author",
    y="kandel_moles",
    color="movement",
    title="Indice de lisibilité Kandel-Moles par auteur (moyenne)",
    labels={
        "kandel_moles": "Score Kandel-Moles",
        "author": "Auteur",
        "movement": "Mouvement littéraire"
    },
    color_discrete_map={"romantic": "#e07b54", "realist": "#4a90d9"}
)

fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

## Loi de Zipf

La loi de Zipf stipule que dans tout corpus textuel, la fréquence d'un mot est inversement proportionnelle à son rang. Autrement dit, le mot le plus fréquent apparaît environ 2 fois plus que le 2ème, 3 fois plus que le 3ème, et ainsi de suite.

Sur un graphique log-log (rang vs fréquence), cette relation donne une droite de pente -1.

Nous calculons les fréquences des mots pour chaque roman, puis nous traçons le graphique rang/fréquence pour vérifier si la loi de Zipf s'applique à notre corpus.

In [51]:
from pyspark.sql.functions import explode, lower, regexp_replace, count
import plotly.express as px

df_one = df_annotated.filter(col("title") == "Pere_Goriot")

df_words = df_one.select(
    "title",
    explode("tokens").alias("token")
).select(
    "title",
    lower(col("token.result")).alias("word")
)

df_words = df_words.filter(col("word").rlike("^[a-z]+$"))

df_freq = df_words.groupBy("word") \
                  .agg(count("*").alias("freq")) \
                  .orderBy(col("freq").desc())

df_freq.show(20)

+----+----+
|word|freq|
+----+----+
| the|5315|
|  to|3052|
|  of|2872|
|   a|2620|
| and|2575|
|  in|1710|
|  he|1513|
|that|1481|
| you|1465|
|   i|1373|
| his|1149|
|  is|1137|
| for| 947|
| was| 937|
|  it| 922|
| her| 894|
|  as| 839|
| not| 701|
|  at| 698|
| had| 698|
+----+----+
only showing top 20 rows



In [52]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

window = Window.orderBy(col("freq").desc())
df_zipf = df_freq.withColumn("rank", row_number().over(window))

df_zipf_pd = df_zipf.toPandas()

fig = px.scatter(
    df_zipf_pd.head(1000),
    x="rank",
    y="freq",
    log_x=True,
    log_y=True,
    title="Loi de Zipf — Père Goriot (Balzac)",
    labels={
        "rank": "Rang (log)",
        "freq": "Fréquence (log)"
    }
)

fig.show()

La courbe log-log obtenue pour *Père Goriot* confirme la loi de Zipf :
on observe une droite de pente négative quasi-parfaite, caractéristique de tout texte naturel. Les premiers rangs (mots très fréquents comme "the", "to", "of") s'écartent légèrement de la droite — c'est normal et observé dans tous les corpus littéraires.

Nous reproduisons maintenant cette analyse sur l'ensemble des 14 romans pour comparer les distributions de fréquences entre auteurs et mouvements.

In [53]:
from pyspark.sql.functions import explode, lower, count, row_number
from pyspark.sql.window import Window

# Pour tous les romans
df_words_all = df_annotated.select(
    "title", "author",
    explode("tokens").alias("token")
).select(
    "title", "author",
    lower(col("token.result")).alias("word")
).filter(col("word").rlike("^[a-z]+$"))

# Fréquences par roman
df_freq_all = df_words_all.groupBy("title", "author", "word") \
                          .agg(count("*").alias("freq"))

# Rang par roman
window = Window.partitionBy("title").orderBy(col("freq").desc())
df_zipf_all = df_freq_all.withColumn("rank", row_number().over(window))

# On garde les 500 premiers rangs par roman
df_zipf_pd = df_zipf_all.filter(col("rank") <= 500).toPandas()

fig = px.line(
    df_zipf_pd.sort_values("rank"),
    x="rank",
    y="freq",
    color="title",
    log_x=True,
    log_y=True,
    title="Loi de Zipf — tous les romans",
    labels={
        "rank": "Rang (log)",
        "freq": "Fréquence (log)",
        "title": "Roman"
    }
)

fig.show()

Le graphique confirme que la loi de Zipf s'applique à l'ensemble du corpus, toutes les courbes sont quasi-parallèles sur l'échelle log-log, ce qui est
caractéristique de tout texte naturel.

Deux observations :

- **Indiana** (Sand) se détache nettement en dessous des autres en raison
  de sa taille bien plus réduite (14 000 mots contre 100 000+ pour les autres)
- Les courbes sont très proches entre elles, la distribution des fréquences
  de mots est similaire pour tous les auteurs, indépendamment du mouvement
  littéraire (romantique ou réaliste)

## Fenêtres glissantes

Les indices de lisibilité calculés sur l'ensemble d'un roman donnent une valeur globale qui masque les variations internes du texte.
Pour observer comment le style évolue au fil du roman, nous calculons FK sur des fenêtres glissantes de taille fixe (500, 1000 et 2000 mots).

Plus la fenêtre est grande, plus l'indice est stable. Une fenêtre
petite révèle davantage de variations locales.

In [54]:
from pyspark.sql.functions import posexplode, floor
import plotly.express as px

# On travaille sur un seul roman pour commencer : Pere Goriot
df_one = df_annotated.filter(col("title") == "Pere_Goriot")

# On explose les tokens avec leur position
df_tokens = df_one.select(
    "title",
    posexplode("tokens").alias("pos", "token")
).select(
    "title",
    col("pos"),
    col("token.result").alias("word")
)

df_tokens.show(5)

+-----------+---+------+
|      title|pos|  word|
+-----------+---+------+
|Pere_Goriot|  0| EBOOK|
|Pere_Goriot|  1|FATHER|
|Pere_Goriot|  2|GORIOT|
|Pere_Goriot|  3|   ***|
|Pere_Goriot|  4|FATHER|
+-----------+---+------+
only showing top 5 rows



In [55]:
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import floor, count, sum as spark_sum

WINDOW_SIZE = 1000

def count_syllables_word(word):
    if word is None:
        return 0
    vowels = "aeiouyAEIOUY"
    count_s = 0
    prev_vowel = False
    for char in word.lower():
        is_vowel = char in vowels
        if is_vowel and not prev_vowel:
            count_s += 1
        prev_vowel = is_vowel
    return max(1, count_s)

syllable_word_udf = udf(count_syllables_word, IntegerType())

df_windows = df_tokens \
    .filter(col("word").rlike("^[a-zA-Z]+$")) \
    .withColumn("window_id", floor(col("pos") / WINDOW_SIZE)) \
    .withColumn("syllables", syllable_word_udf(col("word")))

df_agg = df_windows.groupBy("title", "window_id") \
    .agg(
        count("*").alias("n_words"),
        spark_sum("syllables").alias("n_syllables")
    )

df_sliding = df_agg \
    .withColumn("n_sentences_est", (col("n_words") / 15).cast("double")) \
    .withColumn("flesch_kincaid", spark_round(
        206.835
        - 1.015 * (col("n_words") / col("n_sentences_est"))
        - 84.6  * (col("n_syllables") / col("n_words")), 2
    ))

df_sliding.orderBy("window_id").show(10)

+-----------+---------+-------+-----------+------------------+--------------+
|      title|window_id|n_words|n_syllables|   n_sentences_est|flesch_kincaid|
+-----------+---------+-------+-----------+------------------+--------------+
|Pere_Goriot|        0|    864|       1336|              57.6|         60.79|
|Pere_Goriot|        1|    876|       1331|              58.4|         63.07|
|Pere_Goriot|        2|    862|       1343| 57.46666666666667|          59.8|
|Pere_Goriot|        3|    847|       1359| 56.46666666666667|         55.87|
|Pere_Goriot|        4|    872|       1362| 58.13333333333333|         59.47|
|Pere_Goriot|        5|    852|       1295|              56.8|         63.02|
|Pere_Goriot|        6|    887|       1432| 59.13333333333333|         55.03|
|Pere_Goriot|        7|    836|       1309|55.733333333333334|         59.14|
|Pere_Goriot|        8|    836|       1270|55.733333333333334|         63.09|
|Pere_Goriot|        9|    880|       1386|58.666666666666664|  

In [56]:
df_sliding_pd = df_sliding.orderBy("window_id").toPandas()

fig = px.line(
    df_sliding_pd,
    x="window_id",
    y="flesch_kincaid",
    title="Indice FK sur fenêtres glissantes — Père Goriot (fenêtre = 1000 mots)",
    labels={
        "window_id": "Fenêtre (x1000 mots)",
        "flesch_kincaid": "Score Flesch-Kincaid"
    }
)

fig.add_hline(
    y=df_sliding_pd["flesch_kincaid"].mean(),
    line_dash="dash",
    line_color="red",
    annotation_text="Moyenne"
)

fig.show()

Le score FK de *Père Goriot* oscille entre 55 et 80 autour d'une moyenne stable de 67. La chute en fin de roman correspond probablement à des passages plus denses ou des chapitres de conclusion plus courts.

Nous reproduisons maintenant cette analyse sur l'ensemble des 14 romans pour comparer la stabilité du style entre auteurs et mouvements.

In [37]:
# Sur tous les romans
df_tokens_all = df_annotated.select(
    "title", "author", "movement",
    posexplode("tokens").alias("pos", "token")
).select(
    "title", "author", "movement",
    col("pos"),
    col("token.result").alias("word")
)

df_windows_all = df_tokens_all \
    .filter(col("word").rlike("^[a-zA-Z]+$")) \
    .withColumn("window_id", floor(col("pos") / WINDOW_SIZE)) \
    .withColumn("syllables", syllable_word_udf(col("word")))

df_agg_all = df_windows_all.groupBy("title", "author", "movement", "window_id") \
    .agg(
        count("*").alias("n_words"),
        spark_sum("syllables").alias("n_syllables")
    )

df_sliding_all = df_agg_all \
    .withColumn("n_sentences_est", (col("n_words") / 15).cast("double")) \
    .withColumn("flesch_kincaid", spark_round(
        206.835
        - 1.015 * (col("n_words") / col("n_sentences_est"))
        - 84.6  * (col("n_syllables") / col("n_words")), 2
    ))

df_sliding_pd_all = df_sliding_all.orderBy("title", "window_id").toPandas()

fig = px.line(
    df_sliding_pd_all,
    x="window_id",
    y="flesch_kincaid",
    color="title",
    title="Indice FK sur fenêtres glissantes — tous les romans (fenêtre = 1000 mots)",
    labels={
        "window_id": "Fenêtre (x1000 mots)",
        "flesch_kincaid": "Score Flesch-Kincaid",
        "title": "Roman"
    }
)

fig.show()

Le graphique montre que tous les romans ont un comportement similaire, les scores FK oscillent entre 45 et 85 avec une moyenne autour de 65, indépendamment du mouvement littéraire.

Deux observations :
- Les indices sont **relativement stables** au fil des romans — le style d'un auteur ne change pas drastiquement d'un chapitre à l'autre
- Les variations locales sont plus importantes en début et fin de roman, probablement dues aux chapitres d'introduction et de conclusion qui ont des structures différentes de la narration principale

On passe maintenant à la segmentation **dialogue vs narration**.

## Dialogue vs Narration

La segmentation dialogue/narration permet de caractériser le style d'un auteur. Un roman avec beaucoup de dialogues aura un style plus vivant et direct, tandis qu'un roman narratif sera plus descriptif.

Nous détectons les dialogues comme les passages entre guillemets
(`"..."`) et calculons la proportion de dialogue par rapport à la
narration pour chaque roman.

In [39]:
from pyspark.sql.functions import regexp_extract_all, size, col
from pyspark.sql.types import ArrayType, StringType, IntegerType

def extract_dialog(text):
    if text is None:
        return 0, 0
    import re
    # Guillemets typographiques " " (unicode) ET guillemets standards " "
    dialogs = re.findall(r'[\u201c"]([^\u201d"]*)[\u201d"]', text)
    dialogs += re.findall(r'(?m)^[—–-]\s(.+)$', text)
    dialog_chars = sum(len(d) for d in dialogs)
    total_chars = len(text)
    return int(dialog_chars), int(total_chars)

dialog_schema = StructType([
    StructField("dialog_chars", IntegerType()),
    StructField("total_chars",  IntegerType())
])
dialog_udf = udf(extract_dialog, dialog_schema)

df_dialog = df_annotated \
    .withColumn("dialog_stats", dialog_udf(col("text"))) \
    .withColumn("dialog_chars", col("dialog_stats.dialog_chars")) \
    .withColumn("total_chars", col("dialog_stats.total_chars")) \
    .withColumn("dialog_pct", spark_round(
        col("dialog_chars") / col("total_chars") * 100, 2)) \
    .withColumn("narration_pct", spark_round(
        100 - col("dialog_pct"), 2))

df_dialog.select("author", "title", "movement", "dialog_pct", "narration_pct").show()

+----------+------------------+--------+----------+-------------+
|    author|             title|movement|dialog_pct|narration_pct|
+----------+------------------+--------+----------+-------------+
|    Balzac|       Pere_Goriot|romantic|     47.06|        52.94|
|    Balzac|   Eugenie_Grandet|romantic|      34.4|         65.6|
|     Dumas|      Monte_Cristo|romantic|       0.0|        100.0|
|  Stendhal|     Rouge_et_Noir|romantic|       0.0|        100.0|
|      Sand|           Indiana|romantic|     25.53|        74.47|
|  Bronte_C|         Jane_Eyre|romantic|      41.8|         58.2|
|  Bronte_E| Wuthering_Heights|romantic|     44.35|        55.65|
|     Scott|           Ivanhoe|romantic|     44.44|        55.56|
|  Flaubert|     Madame_Bovary| realist|     17.86|        82.14|
|      Zola|          Germinal| realist|       0.0|        100.0|
|      Zola|         Assommoir| realist|     13.17|        86.83|
|   Dickens|Great_Expectations| realist|     27.93|        72.07|
|   Dicken

In [41]:
import plotly.express as px
df_dialog_pd = df_dialog.select("author", "title", "movement", "dialog_pct", "narration_pct").toPandas()
df_dialog_avg = df_dialog_pd.groupby(["author", "movement"], as_index=False).agg(
    dialog_pct=("dialog_pct", "mean"),
    narration_pct=("narration_pct", "mean")
).round(2)

df_melted = df_dialog_avg.melt(
    id_vars=["author", "movement"],
    value_vars=["dialog_pct", "narration_pct"],
    var_name="type",
    value_name="percentage"
)

df_melted["type"] = df_melted["type"].map({
    "dialog_pct": "Dialogue",
    "narration_pct": "Narration"
})

fig = px.bar(
    df_melted.sort_values("author"),
    x="author",
    y="percentage",
    color="type",
    title="Proportion dialogue vs narration par auteur (moyenne)",
    labels={
        "percentage": "Pourcentage (%)",
        "author": "Auteur",
        "type": "Type"
    },
    color_discrete_map={"Dialogue": "#e07b54", "Narration": "#4a90d9"}
)

fig.update_layout(xaxis_tickangle=-45, barmode="stack")
fig.show()

## Dialogue vs Narration

La segmentation dialogue/narration permet de caractériser le style d'un auteur. Un roman avec beaucoup de dialogues aura un style plus vivant et direct, tandis qu'un roman narratif sera plus descriptif.

### Détection des dialogues

La détection des dialogues s'est révélée plus complexe que prévu.
Une première approche avec des guillemets standards `"..."` ne donnait aucun résultat pour la majorité des romans. En inspectant le texte brut, nous avons découvert que Gutenberg utilise des guillemets typographiques unicode `\u201c` et `\u201d` (`"` et `"`) au lieu des guillemets ASCII standards.

Par ailleurs, certains romans français (Germinal, Rouge et Noir,
Monte Cristo) utilisent les tirets `—` pour les dialogues, convention typographique française. Ces romans affichent donc 0% de dialogue détecté, non pas parce qu'ils n'ont pas de dialogues, mais parce que leur format typographique diffère.

Enfin, certains auteurs ayant deux romans dans le corpus (Balzac,
Dickens, Zola), nous calculons la moyenne des proportions par auteur.

### Résultats
Les romantiques anglais (Brontë, Scott, Balzac) ont les proportions de dialogue les plus élevées (40-45%), tandis que les réalistes français (Zola, Flaubert, Maupassant) sont plus narratifs (moins de 20% de dialogue).